# Marmousi Elastic OPT Debug

Small `2DsismoTimeIsoHetero` check using Marmousi-derived `rho`, `vp`, and `vs`. The material variables passed to the equation are `rho`, `lambda`, and `mu` in MKS units.


## 1. Setup


In [ ]:
using Pkg
cd(@__DIR__)
Pkg.activate("../")

try
    using Metal
catch err
    @warn "Metal not loaded; continuing on CPU" err
end

using JLD2
ParamFile = "../config/testparam.csv"

include("../src/batchFiles/batchGPU.jl")
include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")
using .commonBatchs, .planet1D, .GeoPoints

include("../src/flexOPT.jl")
using .flexOPT

include("temporaryHelpers.jl")


## 2. Homogeneous Elastic Smoke Test

This first checks the LHS propagation with a Gaussian initial field. The CFL is intentionally conservative because the current OPT3 elastic stencil can be unstable; if it still blows up, inspect the LHS stencil before testing source `Γ`.


In [ ]:
shape_homo = (41, 41)
dx_homo = 1.0e3
rho0_homo = 2500.0
vp0_homo = 3200.0
vs0_homo = 1800.0

rho_homo = fill(rho0_homo, shape_homo)
vp_homo = fill(vp0_homo, shape_homo)
vs_homo = fill(vs0_homo, shape_homo)
lambda_homo = rho_homo .* vp_homo.^2 .- 2 .* rho_homo .* vs_homo.^2
mu_homo = rho_homo .* vs_homo.^2
models_homo_mks = [rho_homo, lambda_homo, mu_homo]

cfl_homo = 0.08
cfl_info_homo = cfl_diagnostics(vp_homo, (dx_homo, dx_homo, 1.0); cfl_safety=cfl_homo)
dt_homo = cfl_info_homo.suggested_dt_2D
delta_homo_mks = (dx_homo, dx_homo, dt_homo)

# Dimensionless elastic recipe: rho'=1, lambda'=(lambda/rho)*(dt/dx)^2, mu'=(mu/rho)*(dt/dx)^2.
# This is the elastic analogue of acoustic velocity_cfl = v*dt/dx.
rho_homo_nd = ones(Float64, shape_homo)
lambda_homo_nd = (lambda_homo ./ rho_homo) .* (dt_homo / dx_homo)^2
mu_homo_nd = (mu_homo ./ rho_homo) .* (dt_homo / dx_homo)^2
models_homo = [rho_homo_nd, lambda_homo_nd, mu_homo_nd]
delta_homo = (1.0, 1.0, 1.0)

sourcePoint_homo = CartesianIndex(cld(shape_homo[1], 2), cld(shape_homo[2], 2))
Nt_homo = 30
store_every_homo = 2
total_time_homo = Nt_homo * dt_homo
source_f0_homo = min(max(cfl_info_homo.suggested_f0, 0.15), 1 / (8dt_homo))
source_t0_homo = min(12dt_homo, 0.25total_time_homo)
source_amplitude_homo = 1.0

@show delta_homo_mks delta_homo extrema(lambda_homo_nd) extrema(mu_homo_nd)
@show sourcePoint_homo Nt_homo store_every_homo total_time_homo source_f0_homo source_t0_homo


In [ ]:
elasticOPT_homo = build_opt_prepared(
    "2DsismoTimeIsoHeteroForce",
    models_homo,
    delta_homo;
    pointsInSpace=3,
    pointsInTime=3,
    supplementaryOrder=2,
    orderBspace=1,
    orderBtime=1,
    YorderBspace=-1,
    YorderBtime=-1,
    modelName="homogeneous_elastic_OPT",
)
preparedElastic_homo = elasticOPT_homo.prepared
elastic_A_report_homo = implicit_matrix_report(preparedElastic_homo)

@show preparedElastic_homo.spaceShape preparedElastic_homo.NField preparedElastic_homo.NForceField
@show elastic_A_report_homo

rho0_nd_homo = rho_homo_nd[sourcePoint_homo]
lambda0_nd_homo = lambda_homo_nd[sourcePoint_homo]
mu0_nd_homo = mu_homo_nd[sourcePoint_homo]
lhs_diag_homo = elastic_lhs_local_diagnostics(elasticOPT_homo.numOps, sourcePoint_homo, rho0_nd_homo, lambda0_nd_homo, mu0_nd_homo)
@show rho0_nd_homo lambda0_nd_homo mu0_nd_homo
@show lhs_diag_homo.rows[1]
@show lhs_diag_homo.rows[2]
@show lhs_diag_homo.rows[3]
@show lhs_diag_homo.rows[4]
lhs_diag_homo.matrices


In [ ]:
recipe_lhs_symbolic_matrices(elasticOPT_homo.optRec; iExpr=1, iField=1)

In [ ]:
using Symbolics

In [ ]:
semi_11 = recipe_lhs_evaluated_summary(
    elasticOPT_homo.optRec,
    (rho0_nd_homo, lambda0_nd_homo, mu0_nd_homo);
    iExpr=1,
    iField=1,
)

semi_12 = recipe_lhs_evaluated_summary(
    elasticOPT_homo.optRec,
    (rho0_nd_homo, lambda0_nd_homo, mu0_nd_homo);
    iExpr=1,
    iField=2,
)

semi_11

In [ ]:
function symbol_at_pi(summary)
    Dict(row.time_role => sum(((-1)^(i+j)) * row.matrix[i,j]
                              for i in 1:3, j in 1:3)
         for row in summary)
end

symbol_at_pi(semi_11)
symbol_at_pi(semi_12)

In [ ]:
@show semi_11
@show lhs_diag_homo.matrices[(1,1)]

In [ ]:
recette_homo = elasticOPT_homo.optRec["recette"]
Asemi = recette_homo.lhs.Ajiννᶜ
size(Asemi)

In [ ]:
Asemi

In [ ]:
lhs_diag_homo.rows[1].comparison
lhs_diag_homo.rows[2].comparison
lhs_diag_homo.rows[3].comparison
lhs_diag_homo.rows[4].comparison

In [ ]:
lhs_diag_homo.rows[1].worst_stability
lhs_diag_homo.rows[2].worst_stability
lhs_diag_homo.rows[3].worst_stability
lhs_diag_homo.rows[4].worst_stability

In [ ]:
for key in keys(lhs_diag_homo.matrices)
    println("\nBLOCK ", key)
    for m in lhs_diag_homo.matrices[key]
        println(m.time_role)
        display(m.matrix)
    end
end

In [ ]:
@show lhs_diag_homo.matrices[(1,1)]
@show lhs_diag_homo.matrices[(1,2)]
@show lhs_diag_homo.comparisons[(1,1)]
@show lhs_diag_homo.comparisons[(1,2)]

In [ ]:
lhs_diag_homo.comparisons[(1,1)]
lhs_diag_homo.matrices[(1,1)]
lhs_diag_homo.stability[(1,1)][1]

In [ ]:
initial_ux_homo = gaussian_field(shape_homo, sourcePoint_homo; sigma=8.0, amplitude=1.0)
initial_homo = zeros(Float64, shape_homo..., 2)
initial_homo[:, :, 1] .= initial_ux_homo

frames_gauss_homo_full = propagate_linear_frames_with_source(
    preparedElastic_homo,
    Nt_homo;
    initialPast=initial_homo,
    initialPresent=initial_homo,
    store_every=store_every_homo,
    blowup_limit=1e6,
)
ux_gauss_homo = component_frames(frames_gauss_homo_full, 1)
uz_gauss_homo = component_frames(frames_gauss_homo_full, 2)
@show wavefield_snapshot_report(ux_gauss_homo)[end] wavefield_snapshot_report(uz_gauss_homo)[end]
@show maximum(abs, ux_gauss_homo[end]) maximum(abs, uz_gauss_homo[end])

# Source-moment test after the LHS Gaussian test. M11 only here.
timeSignal_homo = source_time_samples(Nt_homo, dt_homo, preparedElastic_homo.timePointsUsedForOneStep; t0=source_t0_homo, f0=source_f0_homo)
sourceFull_homo = point_source_full(preparedElastic_homo, sourcePoint_homo, timeSignal_homo; iForceField=1, amplitude=source_amplitude_homo)
source_peak_index_homo = argmax(abs.(timeSignal_homo))
source_peak_time_homo = (source_peak_index_homo - 1) * dt_homo
source_rhs_homo = source_rhs_diagnostics(preparedElastic_homo, sourceFull_homo; it=min(source_peak_index_homo, Nt_homo))
@show maximum(abs, timeSignal_homo) source_peak_index_homo source_peak_time_homo maximum(abs, sourceFull_homo)
@show source_rhs_homo.b_max source_rhs_homo.b_norm source_rhs_homo.b_argmax

frames_elastic_full_homo = propagate_linear_frames_with_source(
    preparedElastic_homo,
    Nt_homo;
    sourceFull=sourceFull_homo,
    store_every=store_every_homo,
    blowup_limit=1e6,
)
ux_frames_homo = component_frames(frames_elastic_full_homo, 1)
uz_frames_homo = component_frames(frames_elastic_full_homo, 2)

ux_report_homo = wavefield_snapshot_report(ux_frames_homo)
uz_report_homo = wavefield_snapshot_report(uz_frames_homo)
@show length(frames_elastic_full_homo) ux_report_homo[1] ux_report_homo[end]
@show uz_report_homo[1] uz_report_homo[end]


In [ ]:
display(plot_wave_snapshots(ux_gauss_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic OPT Gaussian ux"))
display(plot_wave_snapshots(uz_gauss_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic OPT Gaussian uz"))
plot_wave_snapshots(ux_frames_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic OPT source M11 ux")


In [ ]:
plot_wave_snapshots(uz_frames_homo; sourcePoint=sourcePoint_homo, title="homogeneous elastic OPT source M11 uz")


## 3. Reduced Elastic Marmousi Material


In [ ]:
marmousi = load(joinpath(@__DIR__, "tmp/seismicModelMarmousi.jld2"), "output")

shape = (41, 41)
downsample_step = 8
rho_raw = downsample_center_crop(marmousi.ρ, shape; step=downsample_step)
vp_raw = downsample_center_crop(marmousi.Vpv, shape; step=downsample_step)
vs_raw = downsample_center_crop(marmousi.Vsv, shape; step=downsample_step)

rho, lambda, mu, vp, vs = elastic_lame_from_rho_vp_vs(rho_raw, vp_raw, vs_raw)

# First small linear test: avoid a singular pure-fluid block. Set to 0.0 when testing the free-surface/fluid behavior explicitly.
vs_floor = 500.0
if vs_floor > 0
    vs = max.(vs, vs_floor)
    mu = rho .* vs.^2
    lambda = rho .* vp.^2 .- 2 .* mu
end

dx = 1.0e3 * downsample_step
cfl = 0.08
cfl_info = cfl_diagnostics(vp, (dx, dx, 1.0); cfl_safety=cfl)
dt = cfl_info.suggested_dt_2D
delta_mks = (dx, dx, dt)

# Dimensionless elastic recipe for visible/stable debug propagation.
rho_nd = ones(Float64, shape)
lambda_nd = (lambda ./ rho) .* (dt / dx)^2
mu_nd = (mu ./ rho) .* (dt / dx)^2
models_elastic = [rho_nd, lambda_nd, mu_nd]
delta = (1.0, 1.0, 1.0)

sourcePoint = CartesianIndex(cld(size(rho, 1), 2), cld(size(rho, 2), 2))
Nt = 30
store_every = 2
total_time = Nt * dt

# Keep the source peak inside this deliberately short elastic debug run.
source_f0 = min(max(cfl_info.suggested_f0, 0.08), 1 / (8dt))
source_t0 = min(12dt, 0.25total_time)
source_amplitude = 1.0

@show size(rho) extrema(rho) extrema(vp) extrema(vs) extrema(lambda) extrema(mu)
@show delta_mks delta extrema(lambda_nd) extrema(mu_nd)
@show sourcePoint Nt store_every total_time source_f0 source_t0


## 4. Build 2DsismoTimeIsoHetero OPT


In [ ]:
elasticOPT = build_opt_prepared(
    "2DsismoTimeIsoHetero",
    models_elastic,
    delta;
    pointsInSpace=3,
    pointsInTime=3,
    supplementaryOrder=2,
    orderBspace=1,
    orderBtime=1,
    YorderBspace=-1,
    YorderBtime=-1,
    modelName="Marmousi_elastic_reduced_OPT",
)
preparedElastic = elasticOPT.prepared
elastic_A_report = implicit_matrix_report(preparedElastic)

@show preparedElastic.spaceShape preparedElastic.NpointsSpace preparedElastic.NField preparedElastic.NForceField preparedElastic.timePointsUsedForOneStep
@show size(preparedElastic.A_unknown) nnz(preparedElastic.A_unknown) nnz(preparedElastic.L_known) nnz(preparedElastic.R_force)
@show elastic_A_report

rho0_nd = rho_nd[sourcePoint]
lambda0_nd = lambda_nd[sourcePoint]
mu0_nd = mu_nd[sourcePoint]
lhs_diag_marm = elastic_lhs_local_diagnostics(elasticOPT.numOps, sourcePoint, rho0_nd, lambda0_nd, mu0_nd)
@show rho0_nd lambda0_nd mu0_nd
@show lhs_diag_marm.rows[1]
@show lhs_diag_marm.rows[2]
@show lhs_diag_marm.rows[3]
@show lhs_diag_marm.rows[4]
lhs_diag_marm.matrices


## 5. Marmousi Source And Linear Propagation


In [ ]:
initial_ux = gaussian_field(size(rho), sourcePoint; sigma=8.0, amplitude=1.0)
initial = zeros(Float64, size(rho)..., 2)
initial[:, :, 1] .= initial_ux

frames_gauss_full = propagate_linear_frames_with_source(
    preparedElastic,
    Nt;
    initialPast=initial,
    initialPresent=initial,
    store_every=store_every,
    blowup_limit=1e6,
)
ux_gauss = component_frames(frames_gauss_full, 1)
uz_gauss = component_frames(frames_gauss_full, 2)
@show wavefield_snapshot_report(ux_gauss)[end] wavefield_snapshot_report(uz_gauss)[end]
@show maximum(abs, ux_gauss[end]) maximum(abs, uz_gauss[end])

# Source-moment test after the LHS Gaussian test. M11 only here.
timeSignal = source_time_samples(Nt, dt, preparedElastic.timePointsUsedForOneStep; t0=source_t0, f0=source_f0)
sourceFull = point_source_full(preparedElastic, sourcePoint, timeSignal; iForceField=1, amplitude=source_amplitude)
source_peak_index = argmax(abs.(timeSignal))
source_peak_time = (source_peak_index - 1) * dt
source_rhs = source_rhs_diagnostics(preparedElastic, sourceFull; it=min(source_peak_index, Nt))
@show maximum(abs, timeSignal) source_peak_index source_peak_time maximum(abs, sourceFull)
@show source_rhs.b_max source_rhs.b_norm source_rhs.b_argmax

frames_elastic_full = propagate_linear_frames_with_source(
    preparedElastic,
    Nt;
    sourceFull=sourceFull,
    store_every=store_every,
    blowup_limit=1e6,
)
ux_frames = component_frames(frames_elastic_full, 1)
uz_frames = component_frames(frames_elastic_full, 2)

ux_report = wavefield_snapshot_report(ux_frames)
uz_report = wavefield_snapshot_report(uz_frames)
@show length(frames_elastic_full) ux_report[1] ux_report[end]
@show uz_report[1] uz_report[end]


In [ ]:
display(plot_wave_snapshots(ux_gauss; sourcePoint=sourcePoint, title="Marmousi elastic OPT Gaussian ux"))
display(plot_wave_snapshots(uz_gauss; sourcePoint=sourcePoint, title="Marmousi elastic OPT Gaussian uz"))
plot_wave_snapshots(ux_frames; sourcePoint=sourcePoint, title="Marmousi elastic OPT source M11 ux")


In [ ]:
plot_wave_snapshots(uz_frames; sourcePoint=sourcePoint, title="Marmousi elastic OPT source M11 uz")


## 6. Local Stencil Inspection


In [ ]:
st_elastic_eq1_u1 = operator_stencil_at_point(elasticOPT.numOps, sourcePoint; which=:left, iExpr=1, iField=1)
st_elastic_eq2_u2 = operator_stencil_at_point(elasticOPT.numOps, sourcePoint; which=:left, iExpr=2, iField=2)

stencil_time_summary(st_elastic_eq1_u1), stencil_time_summary(st_elastic_eq2_u2)
